In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
# load the raw general data and fix messy column names
g = pd.read_csv('raw_general_data.csv')
g.columns = g.columns.str.strip()
g = g.rename(columns={'Monthly_Income': 'MonthlyIncome'})

g.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome',
       'Education', 'EducationField', 'EmployeeCount', 'EmployeeID', 'Gender',
       'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome',
       'NumCompaniesWorked', 'Over18', 'PercentSalaryHike', 'StandardHours',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='str')

In [3]:
# fix inconsistent text casing and stray spaces in category columns
text_cols = ['Department', 'BusinessTravel', 'Gender', 'MaritalStatus', 'EducationField', 'JobRole', 'Attrition']
for col in text_cols:
    g[col] = g[col].str.strip().str.title()

g['Department'].unique()

<ArrowStringArray>
['Research & Development', 'Sales', 'Human Resources']
Length: 3, dtype: str

In [4]:
# numbers got saved as messy text, converting them back to proper numeric type
num_cols = ['Age', 'MonthlyIncome', 'PercentSalaryHike', 'YearsAtCompany']
for col in num_cols:
    g[col] = pd.to_numeric(g[col].astype(str).str.strip(), errors='coerce').astype(int)

g[num_cols].dtypes

Age                  int64
MonthlyIncome        int64
PercentSalaryHike    int64
YearsAtCompany       int64
dtype: object

In [5]:
# remove duplicate employee rows and sort by EmployeeID
g = g.drop_duplicates(subset='EmployeeID').sort_values('EmployeeID').reset_index(drop=True)

# these columns are the same value for every employee, no use keeping them
g = g.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])

g.shape

(4410, 21)

In [6]:
# load and clean the employee survey file (self-reported satisfaction scores)
e = pd.read_csv('raw_employee_survey_data.csv')
e = e.rename(columns={'employee_id': 'EmployeeID'})
e = e.drop_duplicates(subset='EmployeeID').sort_values('EmployeeID').reset_index(drop=True)

e.shape

(4410, 4)

In [7]:
# load and clean the manager survey file (manager-given ratings)
m = pd.read_csv('raw_manager_survey_data.csv')
for col in ['JobInvolvement', 'PerformanceRating']:
    m[col] = pd.to_numeric(m[col].astype(str).str.strip(), errors='coerce').astype(int)
m = m.sort_values('EmployeeID').reset_index(drop=True)

m.shape

(4410, 3)

In [8]:
# combine all three cleaned files into one working table using EmployeeID
df = g.merge(e, on='EmployeeID').merge(m, on='EmployeeID')

df.shape

(4410, 26)

In [9]:
########----------EDA Section-------------#########

In [10]:
# how many employees left vs stayed, and what % that is
df['Attrition'].value_counts()

Attrition
No     3699
Yes     711
Name: count, dtype: int64

In [11]:
# attrition rate as a percentage
df['Attrition'].value_counts(normalize=True) * 100

Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64

In [12]:
# attrition rate by department
df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)

Department
Human Resources           30.2
Research & Development    15.7
Sales                     15.0
Name: Attrition, dtype: float64

In [13]:
# how many employees, and how many left, by department
df.groupby('Department')['Attrition'].value_counts()

Department              Attrition
Human Resources         No            132
                        Yes            57
Research & Development  No           2430
                        Yes           453
Sales                   No           1137
                        Yes           201
Name: count, dtype: int64

In [14]:
# create age bands to check attrition pattern by age group
df['AgeGroup'] = pd.cut(df['Age'], bins=[17,25,35,45,55,65], labels=['18-25','26-35','36-45','46-55','56+'])
df.groupby('AgeGroup')['Attrition'].apply(lambda x: (x=='Yes').mean()*100).round(1)

AgeGroup
18-25    35.8
26-35    19.1
36-45     9.2
46-55    11.5
56+      17.0
Name: Attrition, dtype: float64

In [15]:
df['Left'] = (df['Attrition'] == 'Yes').astype(int)

factors = ['EnvironmentSatisfaction','JobSatisfaction','WorkLifeBalance',
           'YearsAtCompany','TotalWorkingYears','YearsWithCurrManager']
for f in factors:
    print(f, round(df[f].corr(df['Left']), 3))

EnvironmentSatisfaction -0.102
JobSatisfaction -0.103
WorkLifeBalance -0.063
YearsAtCompany -0.134
TotalWorkingYears -0.17
YearsWithCurrManager -0.156


In [16]:
# -----------------------## Feature Eng##--------------------

In [17]:
# points for known risk factors: young, low experience, new manager, low satisfaction
df['Risk_Score'] = (
    (df['Age'] <= 25) * 2 +
    (df['TotalWorkingYears'] <= 5) * 2 +
    (df['YearsWithCurrManager'] <= 1) * 2 +
    (df['JobSatisfaction'] <= 2) * 1 +
    (df['EnvironmentSatisfaction'] <= 2) * 1
)

df['Risk_Tier'] = df['Risk_Score'].apply(lambda s: 'High' if s>=6 else ('Medium' if s>=3 else 'Low'))
df['Risk_Tier'].value_counts()

Risk_Tier
Low       3190
Medium    1001
High       219
Name: count, dtype: int64

In [18]:
# does the High tier actually have a higher real attrition rate?
df.groupby('Risk_Tier')['Attrition'].apply(lambda x: (x=='Yes').mean()*100).round(1)

Risk_Tier
High      53.4
Low       11.4
Medium    23.0
Name: Attrition, dtype: float64

In [19]:
# assume replacement cost = 6 months of salary (standard HR estimate)
df['Replacement_Cost'] = df['MonthlyIncome'] * 6

high_risk_cost = df[df['Risk_Tier']=='High']['Replacement_Cost'].sum()
print(high_risk_cost)

91527660
